# 05 · Window functions (Spark)

A **window function** computes a value from a group of rows (the *window*) but **keeps every row**
(GROUP BY would squash them into one).

```
function() OVER (PARTITION BY <groups> ORDER BY <sort> ROWS BETWEEN ... )
```
- `PARTITION BY` = restart for each group (like GROUP BY, but rows stay)
- `ORDER BY` = the order inside the group (needed for ranking, running totals, LAG/LEAD)
- frame (`ROWS BETWEEN ...`) = which rows around the current row to use

You **cannot** use a window function in WHERE. Wrap it in a subquery/CTE and filter outside.
BigQuery (and Oracle 23ai) also has `QUALIFY`. Spark SQL has no QUALIFY.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/05_window_functions.ipynb`](../../spark/practice/05_window_functions.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. ROW_NUMBER vs RANK vs DENSE_RANK  ·  🟢 easy

For department 2 (Sales), show `name`, `salary` and three columns ordered by salary (high first):
`rn` (ROW_NUMBER, ties broken by name), `rnk` (RANK), `drnk` (DENSE_RANK). Look at the 90,000 tie: what does each function do?

*Expected result: 6 rows · columns: `name`, `salary`, `rn`, `rnk`, `drnk`*

**How it works:** All three number rows by salary (high first) inside one window. They differ only on ties: ROW_NUMBER never repeats, RANK repeats then skips, DENSE_RANK repeats without gaps.

In [ ]:
%%sql
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS rn,
       RANK()       OVER (ORDER BY salary DESC)       AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC)       AS drnk
FROM employees
WHERE dept_id = 2
ORDER BY rn

,name,salary,rn,rnk,drnk
0,Henry Clark,140000.00,1,1,1
1,Irene Lewis,90000.00,2,2,2
2,Jack Walker,90000.00,3,2,2
3,Karen Hall,90000.00,4,2,2
4,Leo Allen,70000.00,5,5,3
5,Mia Young,65000.00,6,6,4


**DataFrame API:**

In [ ]:
w = Window.orderBy(F.desc("salary"))
res = (employees.filter("dept_id = 2")
       .select("name", "salary",
               F.row_number().over(Window.orderBy(F.desc("salary"), "name")).alias("rn"),
               F.rank().over(w).alias("rnk"),
               F.dense_rank().over(w).alias("drnk"))
       .orderBy("rn"))
res.show()

+-----------+---------+---+---+----+
|       name|   salary| rn|rnk|drnk|
+-----------+---------+---+---+----+
|Henry Clark|140000.00|  1|  1|   1|
|Irene Lewis| 90000.00|  2|  2|   2|
|Jack Walker| 90000.00|  3|  2|   2|
| Karen Hall| 90000.00|  4|  2|   2|
|  Leo Allen| 70000.00|  5|  5|   3|
|  Mia Young| 65000.00|  6|  6|   4|
+-----------+---------+---+---+----+

**Why / traps:** Ties 90k ×3: ROW_NUMBER → 2,3,4 (always different). RANK → 2,2,2 then **5** (skips). DENSE_RANK → 2,2,2 then **3** (no gaps). Most asked window question.

A window with ORDER BY but no PARTITION BY moves all rows to one partition. OK for small data; slow for big data (Spark warns you).

## Q2. Top 2 salary levels per department  ·  🟡 medium

For each department (skip NULL dept_id) show employees whose salary is in the **top 2 different salary values** of that department. Show `dept_id`, `name`, `salary`.

*Expected result: 15 rows · columns: `dept_id`, `name`, `salary`*

**How it works:** 1) DENSE_RANK salaries inside each department (`PARTITION BY dept_id`). 2) Keep rank ≤ 2 in an outer query (or with QUALIFY in BigQuery).

In [ ]:
%%sql
SELECT dept_id, name, salary
FROM (SELECT dept_id, name, salary,
             DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS drnk
      FROM employees
      WHERE dept_id IS NOT NULL) t
WHERE drnk <= 2
ORDER BY dept_id, salary DESC, name

,dept_id,name,salary
0,1,Alice Johnson,250000.00
1,1,Bob Smith,150000.00
2,2,Henry Clark,140000.00
3,2,Irene Lewis,90000.00
4,2,Jack Walker,90000.00
5,2,Karen Hall,90000.00
6,3,Nathan King,130000.00
7,3,Olivia Scott,85000.00
8,3,Paul Adams,85000.00
9,4,Rachel Nelson,135000.00


**DataFrame API:**

In [ ]:
w = Window.partitionBy("dept_id").orderBy(F.desc("salary"))
res = (employees.filter(F.col("dept_id").isNotNull())
       .withColumn("drnk", F.dense_rank().over(w))
       .filter("drnk <= 2")
       .select("dept_id", "name", "salary")
       .orderBy("dept_id", F.desc("salary"), "name"))
res.show()

+-------+--------------+---------+
|dept_id|          name|   salary|
+-------+--------------+---------+
|      1| Alice Johnson|250000.00|
|      1|     Bob Smith|150000.00|
|      2|   Henry Clark|140000.00|
|      2|   Irene Lewis| 90000.00|
|      2|   Jack Walker| 90000.00|
|      2|    Karen Hall| 90000.00|
|      3|   Nathan King|130000.00|
|      3|  Olivia Scott| 85000.00|
|      3|    Paul Adams| 85000.00|
|      4| Rachel Nelson|135000.00|
|      4|    Sam Carter|100000.00|
|      5|Victor Roberts|105000.00|
|      5|  Wendy Turner| 72000.00|
|      6| Yara Campbell|115000.00|
|      6| Chloe Collins| 80000.00|
+-------+--------------+---------+

**Why / traps:** "Top N **people**" → ROW_NUMBER. "Top N **salaries**, keep ties" → DENSE_RANK. Ask which one!

## Q3. Third highest salary  ·  🟡 medium

Show the **3rd highest different** salary in the company as `third_highest`.

*Expected result: 1 row · columns: `third_highest`*

**How it works:** DENSE_RANK over all salaries (no PARTITION BY); rank 3 = the third different salary. DISTINCT shows it once.

In [ ]:
%%sql
SELECT DISTINCT salary AS third_highest
FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS drnk
      FROM employees) t
WHERE drnk = 3

,third_highest
0,140000.00


**DataFrame API:**

In [ ]:
res = (employees.withColumn("drnk", F.dense_rank().over(Window.orderBy(F.desc("salary"))))
       .filter("drnk = 3").select(F.col("salary").alias("third_highest")).distinct())
res.show()

+-------------+
|third_highest|
+-------------+
|    140000.00|
+-------------+

**Why / traps:** DENSE_RANK + DISTINCT, because several people can share the 3rd salary. Other ways: `OFFSET 2` on distinct salaries; or a correlated COUNT(DISTINCT).

## Q4. Running total of revenue  ·  🟡 medium

For **completed** orders in 2024 (each order once), show `order_date`, `daily_revenue` and `running_total` (sum of all days so far in 2024).

*Expected result: 25 rows · columns: `order_date`, `daily_revenue`, `running_total`*

**How it works:** 1) One row per day (SUM per date). 2) `SUM(...) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING → CURRENT ROW)` adds all days so far.

In [ ]:
%%sql
WITH daily AS (
    SELECT order_date, SUM(amount) AS daily_revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
      AND order_date >= DATE '2024-01-01' AND order_date < DATE '2025-01-01'
    GROUP BY order_date
)
SELECT order_date, daily_revenue,
       SUM(daily_revenue) OVER (ORDER BY order_date
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily
ORDER BY order_date

,order_date,daily_revenue,running_total
0,2024-01-08,82.00,82.00
1,2024-01-13,45.00,127.00
2,2024-01-21,80.00,207.00
3,2024-02-06,140.00,347.00
4,2024-02-07,62.00,409.00
5,2024-02-21,312.00,721.00
6,2024-03-05,306.00,1027.00
7,2024-03-13,40.00,1067.00
8,2024-03-22,30.00,1097.00
9,2024-04-10,165.00,1262.00


**DataFrame API:**

In [ ]:
daily = (orders.dropDuplicates()
         .filter((F.col("status") == "completed") & (F.year("order_date") == 2024))
         .groupBy("order_date").agg(F.sum("amount").alias("daily_revenue")))
w = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
res = daily.withColumn("running_total", F.sum("daily_revenue").over(w)).orderBy("order_date")
res.show()

+----------+-------------+-------------+
|order_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2024-01-08|        82.00|        82.00|
|2024-01-13|        45.00|       127.00|
|2024-01-21|        80.00|       207.00|
|2024-02-06|       140.00|       347.00|
|2024-02-07|        62.00|       409.00|
|2024-02-21|       312.00|       721.00|
|2024-03-05|       306.00|      1027.00|
|2024-03-13|        40.00|      1067.00|
|2024-03-22|        30.00|      1097.00|
|2024-04-10|       165.00|      1262.00|
|2024-05-10|        40.00|      1302.00|
|2024-05-18|       170.00|      1472.00|
|2024-06-27|       400.00|      1872.00|
|2024-07-10|        60.00|      1932.00|
|2024-07-13|        80.00|      2012.00|
|2024-07-31|        47.00|      2059.00|
|2024-09-03|        85.00|      2144.00|
|2024-09-07|        50.00|      2194.00|
|2024-09-13|        65.00|      2259.00|
|2024-09-23|         NULL|      2259.00|
+----------+-------------+-------------+
only showing top

**Why / traps:** Aggregate to one row per day **first**, then run the window. A day with only NULL amounts gives NULL daily revenue; SUM over the window skips it.

## Q5. Salary vs department average  ·  🟢 easy

Show `name`, `dept_id`, `salary`, `dept_avg` (2 decimals) and `diff` = salary − dept_avg (2 decimals) for employees with a department.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `dept_avg`, `diff`*

**How it works:** `AVG(salary) OVER (PARTITION BY dept_id)` puts the department average on every row; subtract it from the salary.

In [ ]:
%%sql
SELECT name, dept_id, salary,
       ROUND(AVG(salary) OVER (PARTITION BY dept_id), 2)          AS dept_avg,
       ROUND(salary - AVG(salary) OVER (PARTITION BY dept_id), 2) AS diff
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,dept_avg,diff
0,Alice Johnson,1,250000.00,134285.71,115714.29
1,Bob Smith,1,150000.00,134285.71,15714.29
2,Carol White,1,120000.00,134285.71,-14285.71
3,David Brown,1,120000.00,134285.71,-14285.71
4,Eva Green,1,110000.00,134285.71,-24285.71
5,Frank Moore,1,95000.00,134285.71,-39285.71
6,Grace Lee,1,95000.00,134285.71,-39285.71
7,Henry Clark,2,140000.00,90833.33,49166.67
8,Irene Lewis,2,90000.00,90833.33,-833.33
9,Jack Walker,2,90000.00,90833.33,-833.33


**DataFrame API:**

In [ ]:
w = Window.partitionBy("dept_id")
res = (employees.filter(F.col("dept_id").isNotNull())
       .withColumn("a", F.avg("salary").over(w))
       .select("name", "dept_id", "salary",
               F.round("a", 2).alias("dept_avg"),
               F.round(F.col("salary") - F.col("a"), 2).alias("diff")))
res.show()

+-------------+-------+---------+---------+---------+
|         name|dept_id|   salary| dept_avg|     diff|
+-------------+-------+---------+---------+---------+
|Alice Johnson|      1|250000.00|134285.71|115714.29|
|    Bob Smith|      1|150000.00|134285.71| 15714.29|
|  Carol White|      1|120000.00|134285.71|-14285.71|
|  David Brown|      1|120000.00|134285.71|-14285.71|
|    Eva Green|      1|110000.00|134285.71|-24285.71|
|  Frank Moore|      1| 95000.00|134285.71|-39285.71|
|    Grace Lee|      1| 95000.00|134285.71|-39285.71|
|  Henry Clark|      2|140000.00| 90833.33| 49166.67|
|  Irene Lewis|      2| 90000.00| 90833.33|  -833.33|
|  Jack Walker|      2| 90000.00| 90833.33|  -833.33|
|   Karen Hall|      2| 90000.00| 90833.33|  -833.33|
|    Leo Allen|      2| 70000.00| 90833.33|-20833.33|
|    Mia Young|      2| 65000.00| 90833.33|-25833.33|
|  Nathan King|      3|130000.00| 90000.00| 40000.00|
| Olivia Scott|      3| 85000.00| 90000.00| -5000.00|
|   Paul Adams|      3| 8500

**Why / traps:** With PARTITION BY and **no** ORDER BY, the window is the whole group: every row gets the group total/average.

## Q6. Share of department payroll  ·  🟢 easy

Show `name`, `dept_id`, `salary` and `pct_of_dept` = salary as % of the department's total salary, rounded to 1 decimal. Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `pct_of_dept`*

**How it works:** `SUM(salary) OVER (PARTITION BY dept_id)` = department total on each row; salary / total × 100 = share.

In [ ]:
%%sql
SELECT name, dept_id, salary,
       ROUND(100 * salary / SUM(salary) OVER (PARTITION BY dept_id), 1) AS pct_of_dept
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,pct_of_dept
0,Alice Johnson,1,250000.00,26.6
1,Bob Smith,1,150000.00,16.0
2,Carol White,1,120000.00,12.8
3,David Brown,1,120000.00,12.8
4,Eva Green,1,110000.00,11.7
5,Frank Moore,1,95000.00,10.1
6,Grace Lee,1,95000.00,10.1
7,Henry Clark,2,140000.00,25.7
8,Irene Lewis,2,90000.00,16.5
9,Jack Walker,2,90000.00,16.5


**DataFrame API:**

In [ ]:
res = (employees.filter(F.col("dept_id").isNotNull())
       .withColumn("pct_of_dept",
                   F.round(100 * F.col("salary") / F.sum("salary").over(Window.partitionBy("dept_id")), 1))
       .select("name", "dept_id", "salary", "pct_of_dept"))
res.show()

+-------------+-------+---------+-----------+
|         name|dept_id|   salary|pct_of_dept|
+-------------+-------+---------+-----------+
|Alice Johnson|      1|250000.00|       26.6|
|    Bob Smith|      1|150000.00|       16.0|
|  Carol White|      1|120000.00|       12.8|
|  David Brown|      1|120000.00|       12.8|
|    Eva Green|      1|110000.00|       11.7|
|  Frank Moore|      1| 95000.00|       10.1|
|    Grace Lee|      1| 95000.00|       10.1|
|  Henry Clark|      2|140000.00|       25.7|
|  Irene Lewis|      2| 90000.00|       16.5|
|  Jack Walker|      2| 90000.00|       16.5|
|   Karen Hall|      2| 90000.00|       16.5|
|    Leo Allen|      2| 70000.00|       12.8|
|    Mia Young|      2| 65000.00|       11.9|
|  Nathan King|      3|130000.00|       36.1|
| Olivia Scott|      3| 85000.00|       23.6|
|   Paul Adams|      3| 85000.00|       23.6|
|  Quinn Baker|      3| 60000.00|       16.7|
|Rachel Nelson|      4|135000.00|       31.3|
|   Sam Carter|      4|100000.00| 

## Q7. Days since the previous order (LAG)  ·  🟡 medium

For each customer's orders (each order once), show `customer_id`, `order_id`, `order_date`, `prev_order_date` and `days_since_prev` (whole days). The first order of a customer has NULLs.

*Expected result: 104 rows · columns: `customer_id`, `order_id`, `order_date`, `prev_order_date`, `days_since_prev`*

**How it works:** `LAG(order_date)` over each customer's orders (sorted by date) gives the previous order date; subtract the two dates for the gap.

In [ ]:
%%sql
SELECT customer_id, order_id, order_date,
       LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_order_date,
       DATEDIFF(order_date,
                LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id))
           AS days_since_prev
FROM (SELECT DISTINCT order_id, customer_id, order_date FROM orders)

,customer_id,order_id,order_date,prev_order_date,days_since_prev
0,1,1026,2024-05-10,None,NULL
1,2,1104,2025-12-28,None,NULL
2,3,1003,2023-06-25,None,NULL
3,4,1065,2025-05-04,None,NULL
4,5,1057,2025-03-21,None,NULL
5,6,1034,2024-08-04,None,NULL
6,7,1002,2023-06-13,None,NULL
7,8,1022,2024-03-31,None,NULL
8,9,1056,2025-03-03,None,NULL
9,9,1071,2025-06-16,2025-03-03,105.0


**DataFrame API:**

In [ ]:
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
res = (orders.select("order_id", "customer_id", "order_date").distinct()
       .withColumn("prev_order_date", F.lag("order_date").over(w))
       .withColumn("days_since_prev", F.datediff("order_date", "prev_order_date"))
       .select("customer_id", "order_id", "order_date", "prev_order_date", "days_since_prev"))
res.show()

+-----------+--------+----------+---------------+---------------+
|customer_id|order_id|order_date|prev_order_date|days_since_prev|
+-----------+--------+----------+---------------+---------------+
|          1|    1026|2024-05-10|           NULL|           NULL|
|          2|    1104|2025-12-28|           NULL|           NULL|
|          3|    1003|2023-06-25|           NULL|           NULL|
|          4|    1065|2025-05-04|           NULL|           NULL|
|          5|    1057|2025-03-21|           NULL|           NULL|
|          6|    1034|2024-08-04|           NULL|           NULL|
|          7|    1002|2023-06-13|           NULL|           NULL|
|          8|    1022|2024-03-31|           NULL|           NULL|
|          9|    1056|2025-03-03|           NULL|           NULL|
|          9|    1071|2025-06-16|     2025-03-03|            105|
|         10|    1076|2025-07-10|           NULL|           NULL|
|         10|    1093|2025-09-28|     2025-07-10|             80|
|         

**Why / traps:** `LAG(col, n, default)` = value n rows **before**. `LEAD` = n rows **after**.

Date difference: `DATEDIFF(later, earlier)`. Subquery alias is optional in Spark and BigQuery; Oracle allows no alias too.

## Q8. Month-over-month change  ·  🔴 hard

Monthly revenue of completed orders (each order once). Show `month` (first day of the month), `revenue`, `prev_revenue`, and `pct_change` = (revenue − prev) / prev × 100, 1 decimal.

*Expected result: 27 rows · columns: `month`, `revenue`, `prev_revenue`, `pct_change`*

**How it works:** 1) CTE: revenue per month. 2) `LAG(revenue)` = last month's value. 3) (this − last) / last × 100.

In [ ]:
%%sql
WITH monthly AS (
    SELECT TRUNC(order_date, 'MM') AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
    GROUP BY TRUNC(order_date, 'MM')
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
       ROUND(100 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 1) AS pct_change
FROM monthly
ORDER BY month

,month,revenue,prev_revenue,pct_change
0,2023-03-01,135.00,None,None
1,2023-06-01,115.00,135.00,-14.8
2,2023-08-01,10.00,115.00,-91.3
3,2023-11-01,155.00,10.00,1450.0
4,2023-12-01,102.00,155.00,-34.2
5,2024-01-01,207.00,102.00,102.9
6,2024-02-01,514.00,207.00,148.3
7,2024-03-01,376.00,514.00,-26.8
8,2024-04-01,165.00,376.00,-56.1
9,2024-05-01,210.00,165.00,27.3


**DataFrame API:**

In [ ]:
monthly = (orders.dropDuplicates().filter(F.col("status") == "completed")
           .groupBy(F.trunc("order_date", "MM").alias("month"))
           .agg(F.sum("amount").alias("revenue")))
w = Window.orderBy("month")
res = (monthly.withColumn("prev_revenue", F.lag("revenue").over(w))
       .withColumn("pct_change",
                   F.round(100 * (F.col("revenue") - F.col("prev_revenue")) / F.col("prev_revenue"), 1))
       .orderBy("month"))
res.show()

+----------+-------+------------+----------+
|     month|revenue|prev_revenue|pct_change|
+----------+-------+------------+----------+
|2023-03-01| 135.00|        NULL|      NULL|
|2023-06-01| 115.00|      135.00|     -14.8|
|2023-08-01|  10.00|      115.00|     -91.3|
|2023-11-01| 155.00|       10.00|    1450.0|
|2023-12-01| 102.00|      155.00|     -34.2|
|2024-01-01| 207.00|      102.00|     102.9|
|2024-02-01| 514.00|      207.00|     148.3|
|2024-03-01| 376.00|      514.00|     -26.8|
|2024-04-01| 165.00|      376.00|     -56.1|
|2024-05-01| 210.00|      165.00|      27.3|
|2024-06-01| 400.00|      210.00|      90.5|
|2024-07-01| 187.00|      400.00|     -53.3|
|2024-09-01| 200.00|      187.00|       7.0|
|2024-10-01|  40.00|      200.00|     -80.0|
|2024-11-01| 472.00|       40.00|    1080.0|
|2025-01-01| 585.00|      472.00|      23.9|
|2025-02-01| 265.00|      585.00|     -54.7|
|2025-03-01| 215.00|      265.00|     -18.9|
|2025-04-01| 155.00|      215.00|     -27.9|
|2025-05-0

**Why / traps:** Months with no completed orders are **missing**, so LAG compares with an older month. To be exact, build a full month calendar first (topic 08). Division by zero: BigQuery has `SAFE_DIVIDE`; elsewhere use `NULLIF(prev, 0)`.

First day of month: `TRUNC(d, 'MM')` (returns a date). `DATE_TRUNC('MONTH', d)` returns a timestamp.

## Q9. 3-month moving average  ·  🟡 medium

Same monthly revenue. Show `month`, `revenue`, and `moving_avg_3` = average of this month and the 2 months before it (by row), 2 decimals.

*Expected result: 27 rows · columns: `month`, `revenue`, `moving_avg_3`*

**How it works:** Average over a frame of 3 rows: `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = this month and the 2 rows before.

In [ ]:
%%sql
WITH monthly AS (
    SELECT TRUNC(order_date, 'MM') AS month, SUM(amount) AS revenue
    FROM (SELECT DISTINCT order_id, order_date, amount, status FROM orders) o
    WHERE status = 'completed'
    GROUP BY TRUNC(order_date, 'MM')
)
SELECT month, revenue,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM monthly
ORDER BY month

,month,revenue,moving_avg_3
0,2023-03-01,135.00,135.00
1,2023-06-01,115.00,125.00
2,2023-08-01,10.00,86.67
3,2023-11-01,155.00,93.33
4,2023-12-01,102.00,89.00
5,2024-01-01,207.00,154.67
6,2024-02-01,514.00,274.33
7,2024-03-01,376.00,365.67
8,2024-04-01,165.00,351.67
9,2024-05-01,210.00,250.33


**DataFrame API:**

In [ ]:
monthly = (orders.dropDuplicates().filter(F.col("status") == "completed")
           .groupBy(F.trunc("order_date", "MM").alias("month"))
           .agg(F.sum("amount").alias("revenue")))
w = Window.orderBy("month").rowsBetween(-2, Window.currentRow)
res = monthly.withColumn("moving_avg_3", F.round(F.avg("revenue").over(w), 2)).orderBy("month")
res.show()

+----------+-------+------------+
|     month|revenue|moving_avg_3|
+----------+-------+------------+
|2023-03-01| 135.00|      135.00|
|2023-06-01| 115.00|      125.00|
|2023-08-01|  10.00|       86.67|
|2023-11-01| 155.00|       93.33|
|2023-12-01| 102.00|       89.00|
|2024-01-01| 207.00|      154.67|
|2024-02-01| 514.00|      274.33|
|2024-03-01| 376.00|      365.67|
|2024-04-01| 165.00|      351.67|
|2024-05-01| 210.00|      250.33|
|2024-06-01| 400.00|      258.33|
|2024-07-01| 187.00|      265.67|
|2024-09-01| 200.00|      262.33|
|2024-10-01|  40.00|      142.33|
|2024-11-01| 472.00|      237.33|
|2025-01-01| 585.00|      365.67|
|2025-02-01| 265.00|      440.67|
|2025-03-01| 215.00|      355.00|
|2025-04-01| 155.00|      211.67|
|2025-05-01| 300.00|      223.33|
+----------+-------+------------+
only showing top 20 rows

**Why / traps:** `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = 3 rows. The first 2 months average fewer rows.

## Q10. Salary quartiles (NTILE)  ·  🟢 easy

Split all employees into 4 groups by salary (highest first): `name`, `salary`, `quartile` (1–4). Break ties by `emp_id`.

*Expected result: 30 rows · columns: `name`, `salary`, `quartile`*

**How it works:** `NTILE(4)` sorts the rows and cuts them into 4 groups of (almost) the same size.

In [ ]:
%%sql
SELECT name, salary,
       NTILE(4) OVER (ORDER BY salary DESC, emp_id) AS quartile
FROM employees

,name,salary,quartile
0,Alice Johnson,250000.00,1
1,Bob Smith,150000.00,1
2,Henry Clark,140000.00,1
3,Rachel Nelson,135000.00,1
4,Nathan King,130000.00,1
5,Carol White,120000.00,1
6,David Brown,120000.00,1
7,Yara Campbell,115000.00,1
8,Eva Green,110000.00,2
9,Victor Roberts,105000.00,2


**DataFrame API:**

In [ ]:
res = employees.select("name", "salary",
                       F.ntile(4).over(Window.orderBy(F.desc("salary"), "emp_id")).alias("quartile"))
res.show()

+--------------+---------+--------+
|          name|   salary|quartile|
+--------------+---------+--------+
| Alice Johnson|250000.00|       1|
|     Bob Smith|150000.00|       1|
|   Henry Clark|140000.00|       1|
| Rachel Nelson|135000.00|       1|
|   Nathan King|130000.00|       1|
|   Carol White|120000.00|       1|
|   David Brown|120000.00|       1|
| Yara Campbell|115000.00|       1|
|     Eva Green|110000.00|       2|
|Victor Roberts|105000.00|       2|
|    Sam Carter|100000.00|       2|
| Tina Mitchell| 98000.00|       2|
|     Uma Perez| 98000.00|       2|
|   Frank Moore| 95000.00|       2|
|     Grace Lee| 95000.00|       2|
|   Irene Lewis| 90000.00|       2|
|   Jack Walker| 90000.00|       3|
|    Karen Hall| 90000.00|       3|
|Daniel Stewart| 90000.00|       3|
|  Olivia Scott| 85000.00|       3|
+--------------+---------+--------+
only showing top 20 rows

**Why / traps:** 30 rows / 4 groups: the first 2 groups get 8 rows, the last 2 get 7.

## Q11. Top earner on every row (FIRST_VALUE)  ·  🟡 medium

Show `name`, `dept_id`, `salary` and `top_earner` = the name of the highest-paid person in the same department (ties: first by name). Skip NULL dept_id.

*Expected result: 29 rows · columns: `name`, `dept_id`, `salary`, `top_earner`*

**How it works:** `FIRST_VALUE(name)` over each department sorted by salary high first returns the top earner's name on every row.

In [ ]:
%%sql
SELECT name, dept_id, salary,
       FIRST_VALUE(name) OVER (PARTITION BY dept_id ORDER BY salary DESC, name) AS top_earner
FROM employees
WHERE dept_id IS NOT NULL

,name,dept_id,salary,top_earner
0,Alice Johnson,1,250000.00,Alice Johnson
1,Bob Smith,1,150000.00,Alice Johnson
2,Carol White,1,120000.00,Alice Johnson
3,David Brown,1,120000.00,Alice Johnson
4,Eva Green,1,110000.00,Alice Johnson
5,Frank Moore,1,95000.00,Alice Johnson
6,Grace Lee,1,95000.00,Alice Johnson
7,Henry Clark,2,140000.00,Henry Clark
8,Irene Lewis,2,90000.00,Henry Clark
9,Jack Walker,2,90000.00,Henry Clark


**DataFrame API:**

In [ ]:
w = Window.partitionBy("dept_id").orderBy(F.desc("salary"), "name")
res = (employees.filter(F.col("dept_id").isNotNull())
       .select("name", "dept_id", "salary", F.first("name").over(w).alias("top_earner")))
res.show()

+-------------+-------+---------+-------------+
|         name|dept_id|   salary|   top_earner|
+-------------+-------+---------+-------------+
|Alice Johnson|      1|250000.00|Alice Johnson|
|    Bob Smith|      1|150000.00|Alice Johnson|
|  Carol White|      1|120000.00|Alice Johnson|
|  David Brown|      1|120000.00|Alice Johnson|
|    Eva Green|      1|110000.00|Alice Johnson|
|  Frank Moore|      1| 95000.00|Alice Johnson|
|    Grace Lee|      1| 95000.00|Alice Johnson|
|  Henry Clark|      2|140000.00|  Henry Clark|
|  Irene Lewis|      2| 90000.00|  Henry Clark|
|  Jack Walker|      2| 90000.00|  Henry Clark|
|   Karen Hall|      2| 90000.00|  Henry Clark|
|    Leo Allen|      2| 70000.00|  Henry Clark|
|    Mia Young|      2| 65000.00|  Henry Clark|
|  Nathan King|      3|130000.00|  Nathan King|
| Olivia Scott|      3| 85000.00|  Nathan King|
|   Paul Adams|      3| 85000.00|  Nathan King|
|  Quinn Baker|      3| 60000.00|  Nathan King|
|Rachel Nelson|      4|135000.00|Rachel 

**Why / traps:** Trap: `LAST_VALUE(...) OVER (ORDER BY ...)` gives the **current** row, because the default frame ends at the current row. Add `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

## Q12. Latest order per customer  ·  🟡 medium

For each customer, show their latest order: `customer_id`, `order_id`, `order_date`, `amount`. If two orders are on the same day, take the bigger `order_id`. One row per customer.

*Expected result: 32 rows · columns: `customer_id`, `order_id`, `order_date`, `amount`*

**How it works:** ROW_NUMBER per customer, newest first (`order_date DESC, order_id DESC`), then keep `rn = 1` = exactly one latest row per customer.

In [ ]:
%%sql
SELECT customer_id, order_id, order_date, amount
FROM (SELECT o.*,
             ROW_NUMBER() OVER (PARTITION BY customer_id
                                ORDER BY order_date DESC, order_id DESC) AS rn
      FROM orders o) t
WHERE rn = 1

,customer_id,order_id,order_date,amount
0,1,1026,2024-05-10,40.00
1,2,1104,2025-12-28,135.00
2,3,1003,2023-06-25,70.00
3,4,1065,2025-05-04,40.00
4,5,1057,2025-03-21,90.00
5,6,1034,2024-08-04,42.00
6,7,1002,2023-06-13,45.00
7,8,1022,2024-03-31,225.00
8,9,1071,2025-06-16,12.00
9,10,1093,2025-09-28,69.00


**DataFrame API:**

In [ ]:
w = Window.partitionBy("customer_id").orderBy(F.desc("order_date"), F.desc("order_id"))
res = (orders.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
       .select("customer_id", "order_id", "order_date", "amount"))
res.show()

+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|          1|    1026|2024-05-10| 40.00|
|          2|    1104|2025-12-28|135.00|
|          3|    1003|2023-06-25| 70.00|
|          4|    1065|2025-05-04| 40.00|
|          5|    1057|2025-03-21| 90.00|
|          6|    1034|2024-08-04| 42.00|
|          7|    1002|2023-06-13| 45.00|
|          8|    1022|2024-03-31|225.00|
|          9|    1071|2025-06-16| 12.00|
|         10|    1093|2025-09-28| 69.00|
|         11|    1070|2025-06-03| 65.00|
|         12|    1063|2025-04-18| 60.00|
|         13|    1031|2024-07-10| 60.00|
|         14|    1079|2025-07-15|  NULL|
|         15|    1077|2025-07-12| 20.00|
|         16|    1078|2025-07-13| 45.00|
|         17|    1084|2025-08-01|100.00|
|         18|    1027|2024-05-18|170.00|
|         19|    1039|2024-09-13| 65.00|
|         20|    1024|2024-04-10|120.00|
+-----------+--------+----------+------+
only showing top

**Why / traps:** ROW_NUMBER = exactly one row per group. It also removes the duplicate rows for free. Always add a tie-breaker to ORDER BY, so the result is the same every run.

## Q13. ROWS vs RANGE (ties in a running total)  ·  🔴 hard

Department 2 employees ordered by salary (low → high). Show `name`, `salary`, `rows_total` (frame ROWS ... CURRENT ROW, ties broken by emp_id) and `range_total` (frame RANGE ... CURRENT ROW). Why are they different for the 90,000 tie?

*Expected result: 6 rows · columns: `name`, `salary`, `rows_total`, `range_total`*

**How it works:** ROWS adds one physical row at a time. RANGE adds **all rows with the same salary** at once, so tied rows show the same total.

In [ ]:
%%sql
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary, emp_id ROWS  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
       SUM(salary) OVER (ORDER BY salary         RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total
FROM employees
WHERE dept_id = 2
ORDER BY salary, emp_id

,name,salary,rows_total,range_total
0,Mia Young,65000.00,65000.00,65000.00
1,Leo Allen,70000.00,135000.00,135000.00
2,Irene Lewis,90000.00,225000.00,405000.00
3,Jack Walker,90000.00,315000.00,405000.00
4,Karen Hall,90000.00,405000.00,405000.00
5,Henry Clark,140000.00,545000.00,545000.00


**DataFrame API:**

In [ ]:
rows_w = Window.orderBy("salary", "emp_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
range_w = Window.orderBy("salary").rangeBetween(Window.unboundedPreceding, Window.currentRow)
res = (employees.filter("dept_id = 2")
       .select("name", "salary", "emp_id",
               F.sum("salary").over(rows_w).alias("rows_total"),
               F.sum("salary").over(range_w).alias("range_total"))
       .orderBy("salary", "emp_id").drop("emp_id"))
res.show()

+-----------+---------+----------+-----------+
|       name|   salary|rows_total|range_total|
+-----------+---------+----------+-----------+
|  Mia Young| 65000.00|  65000.00|   65000.00|
|  Leo Allen| 70000.00| 135000.00|  135000.00|
|Irene Lewis| 90000.00| 225000.00|  405000.00|
|Jack Walker| 90000.00| 315000.00|  405000.00|
| Karen Hall| 90000.00| 405000.00|  405000.00|
|Henry Clark|140000.00| 545000.00|  545000.00|
+-----------+---------+----------+-----------+

**Why / traps:** ROWS counts physical rows. RANGE includes **all rows with the same ORDER BY value** (peers). The default frame with ORDER BY is RANGE: that is why running totals jump on ties.